# 🎯 [K15_M05] 점검 이력 요약하기
### 실습 솔루션 & 정답 해설 (Google Colab / Jupyter)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/K15/k15_m05_solution.ipynb)

---

## 📌 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 / 과제** | 군수시설·자산관리 (Level 1) / K15 군 시설 정보 검색 |
| **미션** | 미션 05. 점검 이력 요약하기 (소요시간: 약 1.5시간) |
| **실습 유형** | 조건 제어 프롬프트 엔지니어링 및 팩트체크 검증 (HITL) |

> **⚠️ [교육용 가상 자료 원칙]**
> 본 실습의 모든 자료는 안전 규정에 따라 **교육용 가상 부대 자료(가상 한빛대대)**로 구성되어 있습니다.
> 실제 군사 기밀이나 부대 세부 인적·작전 정보는 외부 AI 도구에 절대 입력하지 않습니다.


In [ ]:
# ============================================================================
# [Colab 환경 설정] 데이터 분석 및 프롬프트 테스트 환경 준비
# ============================================================================
!pip install -q pandas numpy tabulate

import pandas as pd
import numpy as np
import json
import re

# 가상 한빛대대 데이터셋 인메모리 로드 (Colab 환경 독립 실행 보장)
schedule_raw = [
    {"facility": "대대 강당", "day": "월", "09-11": "1중대 교육", "11-13": "1중대 교육", "14-16": "훈련 평가", "16-18": "빈", "19-21": "빈"},
    {"facility": "대대 강당", "day": "화", "09-11": "2중대 교육", "11-13": "빈", "14-16": "3중대 교육", "16-18": "빈", "19-21": "합창 동아리"},
    {"facility": "대대 강당", "day": "수", "09-11": "3중대 교육", "11-13": "3중대 교육", "14-16": "빈", "16-18": "정훈 방송", "19-21": "빈"},
    {"facility": "대대 강당", "day": "목", "09-11": "1중대 안전교육", "11-13": "빈", "14-16": "2중대 교육", "16-18": "2중대 교육", "19-21": "빈"},
    {"facility": "대대 강당", "day": "금", "09-11": "월례 조회", "11-13": "점검(음향)", "14-16": "빈", "16-18": "빈", "19-21": "영화 상영"},
    {"facility": "대대 강당", "day": "토", "09-11": "빈", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "영화 상영"},
    {"facility": "대대 강당", "day": "일", "09-11": "빈", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 A", "day": "월", "09-11": "분대장 회의", "11-13": "빈", "14-16": "분대장 회의", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 A", "day": "화", "09-11": "빈", "11-13": "인사과 상담", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 A", "day": "수", "09-11": "빈", "11-13": "빈", "14-16": "1중대 토의", "16-18": "1중대 토의", "19-21": "빈"},
    {"facility": "세미나실 A", "day": "목", "09-11": "2중대 토의", "11-13": "2중대 토의", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 A", "day": "금", "09-11": "빈", "11-13": "빈", "14-16": "3중대 토의", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 A", "day": "토", "09-11": "독서 모임", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 A", "day": "일", "09-11": "빈", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 B", "day": "월", "09-11": "빈", "11-13": "교육 준비", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 B", "day": "화", "09-11": "점검(빔프로젝터)", "11-13": "점검(빔프로젝터)", "14-16": "점검(빔프로젝터)", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 B", "day": "수", "09-11": "3중대 토의", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 B", "day": "목", "09-11": "빈", "11-13": "빈", "14-16": "빈", "16-18": "AI 활용 스터디", "19-21": "AI 활용 스터디"},
    {"facility": "세미나실 B", "day": "금", "09-11": "대대 간부 회의", "11-13": "대대 간부 회의", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 B", "day": "토", "09-11": "빈", "11-13": "자격증 스터디", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 B", "day": "일", "09-11": "빈", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "체력단련실(단체)", "day": "월", "09-11": "체력검정 연습(1·2중대)", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "체력단련실(단체)", "day": "화", "09-11": "체력검정 연습(3중대)", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "체력단련실(단체)", "day": "수", "09-11": "체력검정 연습(1·2중대)", "11-13": "빈", "14-16": "소독(이용 불가)", "16-18": "빈", "19-21": "빈"},
    {"facility": "체력단련실(단체)", "day": "목", "09-11": "체력검정 연습(3중대)", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "3중대 단체 운동"},
    {"facility": "체력단련실(단체)", "day": "금", "09-11": "체력검정 연습(1·2중대)", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "체력단련실(단체)", "day": "토", "09-11": "빈", "11-13": "빈", "14-16": "대대 체육 연습", "16-18": "대대 체육 연습", "19-21": "빈"},
    {"facility": "체력단련실(단체)", "day": "일", "09-11": "빈", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "빈"}
]

maintenance_raw = [
    {"id": "H01", "date": "08-03", "facility": "세미나실 B", "type": "고장신고", "content": "빔프로젝터 화면 깜빡임", "dept": "군수과", "status": "완료", "days": 2},
    {"id": "H02", "date": "08-10", "facility": "대대 강당", "type": "정기점검", "content": "음향 설비 점검", "dept": "군수과", "status": "완료", "days": 1},
    {"id": "H03", "date": "08-17", "facility": "체력단련실", "type": "고장신고", "content": "러닝머신 1호 벨트 이탈", "dept": "군수과", "status": "완료", "days": 4},
    {"id": "H04", "date": "08-24", "facility": "세미나실 A", "type": "고장신고", "content": "난방기 소음", "dept": "군수과", "status": "완료", "days": 3},
    {"id": "H05", "date": "08-31", "facility": "대대 강당", "type": "고장신고", "content": "무선 마이크 수신 불량", "dept": "정보통신실", "status": "완료", "days": 2},
    {"id": "H06", "date": "09-02", "facility": "세미나실 B", "type": "고장신고", "content": "빔프로젝터 전원 꺼짐", "dept": "군수과", "status": "완료", "days": 3},
    {"id": "H07", "date": "09-07", "facility": "체력단련실", "type": "정기점검", "content": "운동 기구 안전 점검", "dept": "군수과", "status": "완료", "days": 1},
    {"id": "H08", "date": "09-09", "facility": "사이버지식정보방", "type": "고장신고", "content": "PC 3대 부팅 불가", "dept": "정보통신실", "status": "완료", "days": 2},
    {"id": "H09", "date": "09-14", "facility": "대대 강당", "type": "고장신고", "content": "출입문 잠금장치 불량", "dept": "군수과", "status": "완료", "days": 5},
    {"id": "H10", "date": "09-16", "facility": "세미나실 A", "type": "정기점검", "content": "환기 설비 점검", "dept": "군수과", "status": "완료", "days": 1},
    {"id": "H11", "date": "09-21", "facility": "체력단련실", "type": "고장신고", "content": "러닝머신 1호 벨트 이탈", "dept": "군수과", "status": "완료", "days": 6},
    {"id": "H12", "date": "09-23", "facility": "세미나실 B", "type": "정기점검", "content": "영상 장비 점검", "dept": "군수과", "status": "완료", "days": 1},
    {"id": "H13", "date": "09-28", "facility": "대대 강당", "type": "고장신고", "content": "무선 마이크 수신 불량", "dept": "정보통신실", "status": "완료", "days": 3},
    {"id": "H14", "date": "10-01", "facility": "세미나실 A", "type": "고장신고", "content": "난방기 소음", "dept": "군수과", "status": "완료", "days": 4},
    {"id": "H15", "date": "10-05", "facility": "사이버지식정보방", "type": "정기점검", "content": "PC·네트워크 점검", "dept": "정보통신실", "status": "완료", "days": 1},
    {"id": "H16", "date": "10-08", "facility": "세미나실 B", "type": "고장신고", "content": "빔프로젝터 화면 깜빡임", "dept": "군수과", "status": "완료", "days": 4},
    {"id": "H17", "date": "10-13", "facility": "체력단련실", "type": "고장신고", "content": "샤워실 배수 불량", "dept": "군수과", "status": "완료", "days": 2},
    {"id": "H18", "date": "10-15", "facility": "대대 강당", "type": "정기점검", "content": "조명·음향 점검", "dept": "군수과", "status": "완료", "days": 1},
    {"id": "H19", "date": "10-19", "facility": "세미나실 B", "type": "고장신고", "content": "에어컨 냉기 약함", "dept": "군수과", "status": "처리 중", "days": np.nan},
    {"id": "H20", "date": "10-21", "facility": "체력단련실", "type": "고장신고", "content": "러닝머신 1호 벨트 이탈", "dept": "군수과", "status": "처리 중", "days": np.nan},
    {"id": "H21", "date": "10-22", "facility": "대대 강당", "type": "고장신고", "content": "무선 마이크 수신 불량", "dept": "정보통신실", "status": "접수", "days": np.nan},
    {"id": "H22", "date": "10-23", "facility": "세미나실 A", "type": "정기점검", "content": "난방기 점검", "dept": "군수과", "status": "완료", "days": 1}
]

df_schedule = pd.DataFrame(schedule_raw)
df_maintenance = pd.DataFrame(maintenance_raw)

print("✅ [K15] 가상 한빛대대 실습 환경 및 검산용 데이터프레임 로드 완료!")
print(f"- 1주 시설 예약 현황표: {len(df_schedule)}행 (4개 시설 × 7일)")
print(f"- 시설 점검 및 고장 이력: {len(df_maintenance)}건 (정기점검 7건, 고장신고 15건)")


---

## 🎯 미션 05. 점검 이력 요약하기
**소요시간**: 약 1.5시간 | **난이도**: 초급 (Level 1) | **유형**: AI 활용형

### 📋 미션 개요 & 핵심 학습 목표
**[미션 개요]**
<p>22건의 시설 점검 및 고장 이력 데이터를 시설별로 구분하여 정기점검과 고장 신고 현황을 신속하게 파악할 수 있도록 요약합니다. AI를 활용해 시설별 건수, 최근 점검일, 현재 진행 중인 미완료 건(접수·처리 중)을 추출하고 실제 원문 건수를 직접 세어 일치 여부를 검증합니다. 방대한 시설 관리 이력을 한눈에 파악 가능한 행정 요약 정보로 재구성하는 역량을 기릅니다.</p>

**[학습 목표 및 체득 역량]**
<p>점검 이력표 22건을 시설별로 분류하여 정기점검 및 고장 신고 건수, 최근 점검일, 미완료 건을 요약하고 실제 건수를 직접 대조합니다. <strong>이를 통해 방대한 이력 데이터에서 핵심 행정 지표를 빠르게 추출하고, AI 요약 결과의 수치 정확성을 교차 검증하는 데이터 요약 역량을 익힙니다.</strong></p>

### 🛠️ 세부 실습 활동 (GuideSteps)
#### 🔹 Step 1: 시설별로 정기점검과 고장 신고를 나눈다.
<h4>이 단계의 역할</h4><p>8~10월 시설 점검 및 고장 신고 이력 22건(H01~H22)을 시설별로 분류하고 '정기점검'과 '고장신고'로 이원화하여 집계합니다.</p>
<h4>해야 할 일</h4><ol>
<li><p>22건의 이력 데이터를 AI에 입력하고 4개 시설(강당, 세미나실 A, 세미나실 B, 체력단련실, 사이버지식정보방 포함)별로 그룹화합니다.</p></li>
<li><p>각 시설마다 정기점검 건수와 고장신고 건수를 분리 집계하게 합니다.</p></li>
<li><p>정기점검 총합 7건, 고장신고 총합 15건(전체 22건)과 일치하는지 확인합니다.</p></li>
</ol>
<h4>💡 AI 이력 집계 프롬프트 예시</h4><blockquote><p><strong>[시설별 이력 집계 질의]:</strong> "아래 시설 점검·고장 이력 22건을 바탕으로, 각 시설별로 '정기점검' 건수와 '고장신고' 건수를 분리 집계한 요약 테이블을 만들어 줘. 합계가 총 22건이 맞는지도 계산해 줘: [점검 이력 22건 텍스트]"</p></blockquote>
<h4>🔍 검증 포인트 (원문 대조 및 팩트체크)</h4><ul>
<li><strong>총합 일치:</strong> 정기점검 7건 + 고장신고 15건 = 총 22건이 오차 없이 집계되었는가?</li>
<li><strong>시설별 정합성:</strong> 사이버지식정보방(H08, H15) 2건을 포함하여 각 시설별 배정이 올바른가?</li>
</ul>
<h4>핵심 확인 지표</h4><blockquote><p>시설별 정기점검/고장신고 건수 분류 집계표 완성</p></blockquote>
<h4>학습자가 제출해야 하는 내용</h4><ul><li><p>시설별 점검·고장 이력 집계표</p></li></ul>

#### 🔹 Step 2: 최근 점검일과 미완료 건을 뽑는다.
<h4>이 단계의 역할</h4><p>각 시설별 '가장 최근 점검일자'와 현재까지 조치가 끝나지 않은 '미완료 건(접수·처리 중)'을 정확하게 추출합니다.</p>
<h4>해야 할 일</h4><ol>
<li><p>시설별로 가장 늦은 일자(최근 일자)를 역순 정렬하여 최근 점검일을 뽑아냅니다.</p></li>
<li><p>상태 열에서 '처리 중' 또는 '접수'로 기재되어 소요일이 '-'인 미완료 건을 추출합니다.</p></li>
<li><p>미완료 3건(H19 세미나실 B 에어컨 냉기 약함, H20 체력단련실 러닝머신 1호, H21 강당 무선 마이크)이 정확히 식별되었는지 검토합니다.</p></li>
</ol>
<h4>💡 AI 미완료 건 추출 질의 예시</h4><blockquote><p><strong>[미완료 현황 추출 질의]:</strong> "제공된 점검 이력에서 아직 수리가 완료되지 않고 '처리 중'이거나 '접수' 상태인 모든 건의 일련번호(H번호), 일자, 시설, 고장 내용, 담당 부서를 리스트업해 줘."</p></blockquote>
<h4>🔍 검증 포인트 (원문 대조 및 팩트체크)</h4><ul>
<li><strong>미완료 건수:</strong> H19, H20, H21 총 3건이 정확하게 적출되었는가? (완료된 H22는 제외되었는가?)</li>
<li><strong>상태 표기:</strong> H19와 H20은 '처리 중', H21은 '접수' 상태가 왜곡 없이 표기되었는가?</li>
</ul>
<h4>핵심 확인 지표</h4><blockquote><p>미완료 3건 목록 및 시설별 최근 점검일 정리표</p></blockquote>
<h4>학습자가 제출해야 하는 내용</h4><ul><li><p>미완료 건 목록표 및 시설별 최근 점검 현황</p></li></ul>

#### 🔹 Step 3: 건수를 직접 센다.
<h4>이 단계의 역할</h4><p>AI가 추출한 요약 건수 및 수치를 원본 22건 장부와 한 건씩 대조하여 수작업으로 직접 검산(Cross-Check)합니다.</p>
<h4>해야 할 일</h4><ol>
<li><p>AI가 제시한 시설별 정기점검 건수, 고장신고 건수를 출력물 또는 화면 원본과 대조하며 바를 정(正) 자로 직접 계수합니다.</p></li>
<li><p>고장 신고 처리 완료 건(12건)의 소요일 합계 및 평균 소요일(약 3.3일)을 수작업으로 계산합니다.</p></li>
<li><p>AI 계산 수치와 수작업 검산 수치 간에 오차가 있는지 확인하고 불일치 발생 시 원인을 규명합니다.</p></li>
</ol>
<h4>💡 AI 평균 소요일 검산 질의 예시</h4><blockquote><p><strong>[평균 소요일 계산 질의]:</strong> "상태가 '완료'인 고장신고 12건의 소요일을 모두 나열하고, 그 총합과 평균 소요일을 소수점 첫째 자리까지 계산해 줘."</p></blockquote>
<h4>🔍 검증 포인트 (원문 대조 및 팩트체크)</h4><ul>
<li><strong>수기 검산 일치:</strong> 완료 12건 소요일 합계(39일) 및 평균(3.25 -> 3.3일)이 검산 결과와 일치하는가?</li>
<li><strong>미완료 건 제외:</strong> 소요일이 '-'인 미완료 3건이 평균 계산 분모에서 정확히 제외되었는가?</li>
</ul>
<h4>핵심 확인 지표</h4><blockquote><p>AI 요약 건수 및 평균 소요일 수작업 검산 일치율 100%</p></blockquote>
<h4>학습자가 제출해야 하는 내용</h4><ul><li><p>수작업 검산 대조 메모 및 최종 이력 요약표</p></li></ul>

#### 🔹 Step 4: 심화 도전 과제 및 최적화
<h4>이 단계의 역할</h4><p>시설별 평균 처리 소요일을 세분화하여 산출하고, 어떤 시설의 고장 조치가 가장 지연되는지 분석하는 심화 도전을 진행합니다.</p>
<h4>해야 할 일</h4><ol>
<li><p>대대 강당, 세미나실 A, 세미나실 B, 체력단련실 각각의 완료 건 평균 소요일을 산출합니다.</p></li>
<li><p>군수과 처리 건과 정보통신실 처리 건 간의 소요일 편차를 비교합니다.</p></li>
<li><p>분석 결과를 토대로 향후 시설 유지보수 시 우선 집중 관리해야 할 분야를 제언합니다.</p></li>
</ol>
<h4>💡 AI 부서별 처리 속도 분석 예시</h4><blockquote><p><strong>[처리 부서별 비교 질의]:</strong> "군수과에서 처리한 고장 건과 정보통신실에서 처리한 고장 건의 평균 소요일을 각각 계산하고, 처리 지연이 발생하는 주요 원인 시설을 분석해 줘."</p></blockquote>
<h4>🔍 검증 포인트 (원문 대조 및 팩트체크)</h4><ul>
<li><strong>그룹별 정확도:</strong> 군수과와 정보통신실의 완료 건이 정확히 분리 계산되었는가?</li>
<li><strong>수치 검증:</strong> H09(강당 잠금장치 5일), H11(러닝머신 6일) 등 장기 소요 건이 올바르게 반영되었는가?</li>
</ul>
<h4>핵심 확인 지표</h4><blockquote><p>시설 및 부서별 세부 처리 소요일 분석 보고서</p></blockquote>
<h4>학습자가 제출해야 하는 내용</h4><ul><li><p>심화 소요일 분석 메모</p></li></ul>

### 📑 실습 작성 서식 및 체크리스트 (Sample Template)

## 1. 시설별 점검·고장 이력 집계표 (총 22건)
| 시설명 | 정기점검 건수 | 고장신고 건수 | 합계 건수 | 최근 점검 일자 |
| :--- | :--- | :--- | :--- | :--- |
| 대대 강당 | 2건 (H02, H18) | 4건 (H05, H09, H13, H21*) | 6건 | 10-22 (H21) |
| 세미나실 A | 2건 (H10, H22) | 2건 (H04, H14) | 4건 | 10-23 (H22) |
| 세미나실 B | 1건 (H12) | 4건 (H01, H06, H16, H19*) | 5건 | 10-19 (H19) |
| 체력단련실 | 1건 (H07) | 4건 (H03, H11, H17, H20*) | 5건 | 10-21 (H20) |
| 사이버지식정보방 | 1건 (H15) | 1건 (H08) | 2건 | 10-05 (H15) |
| **전체 합계** | **7건** | **15건** | **22건** | - |

## 2. 미완료 건 목록 (3건)
| 관리번호 | 발생일자 | 시설명 | 고장 증상 | 처리부서 | 현재 상태 | 소요일 |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| H19 | 10-19 | 세미나실 B | 에어컨 냉기 약함 | 군수과 | 처리 중 | - |
| H20 | 10-21 | 체력단련실 | 러닝머신 1호 벨트 이탈 | 군수과 | 처리 중 | - |
| H21 | 10-22 | 대대 강당 | 무선 마이크 수신 불량 | 정보통신실 | 접수 | - |

## 3. 고장신고 처리 소요일 수작업 검산표
- **완료된 고장신고 건수**: 12건 (15건 중 미완료 3건 제외)
- **소요일 총합**: 2 + 4 + 3 + 2 + 3 + 2 + 5 + 6 + 3 + 4 + 4 + 2 = **39일**
- **평균 처리 소요일**: 39일 ÷ 12건 = **3.25일 (약 3.3일)**
- [x] AI 계산 결과와 수작업 검산 결과 일치 확인 완료

### 🔍 평가 기준 (PASS 조건)
<h4>미션 판정</h4><p><strong>평가 구분:</strong> 필수</p><p><strong>핵심 평가 기준:</strong> [필수] 정보 요약·수치 교차 검산 — 점검 이력 22건을 정기점검과 고장신고로 분리 요약하고 완료 건 평균 소요일을 수작업으로 교차 검산했는가? (PASS / FAIL)</p><h4>미션 PASS 조건</h4><ul><li><p>시설별 정기점검(7건) 및 고장신고(15건) 이원화 집계 완성</p></li><li><p>미완료 3건(H19, H20, H21) 정확 적출 및 최근 점검일 정리</p></li><li><p>완료 12건 소요일 총합(39일) 및 평균 소요일(약 3.3일) 수작업 검산 일치</p></li></ul><h4>채점자 판정 메모</h4><blockquote><p>AI가 제시한 통계 수치를 맹신하지 않고, 미완료 결측치('-')를 제외한 분모(12)와 분자(39)를 직접 검산하여 수치 왜곡을 바로잡았는지 확인한다.</p></blockquote>


In [ ]:
# [미션 05] 프롬프트 및 실습 결과 기록
mission_05_prompt = """# [미션 05] 시설 점검 이력 요약 및 미완료 건 추출 질의
너는 시설 설비 유지보수 데이터 분석관이다.
제공된 8~10월 시설 점검 및 고장 이력 22건(H01~H22)을 분석하여 다음을 수행하라:
1. 각 시설별로 '정기점검' 건수와 '고장신고' 건수를 분리 집계하고, 가장 최근 점검 일자를 표로 제시하라.
2. 현재 수리가 완료되지 않은 '미완료(접수/처리 중)' 건의 일련번호(H번호), 시설명, 증상, 담당 부서를 추출하라.
3. 수리 완료된 고장신고 건들의 총 소요일과 평균 소요일을 소수점 둘째 자리까지 계산하라.
"""

mission_05_result = """### 📋 시설별 점검·고장 이력 집계표 및 미완료 건 리포트

#### 1. 시설별 점검·고장 이력 집계표 (총 22건)
| 시설명 | 정기점검 건수 | 고장신고 건수 | 합계 건수 | 최근 점검 일자 |
| :--- | :---: | :---: | :---: | :---: |
| 대대 강당 | 2건 (H02, H18) | 4건 (H05, H09, H13, H21*) | **6건** | 10-22 (H21) |
| 세미나실 A | 2건 (H10, H22) | 2건 (H04, H14) | **4건** | 10-23 (H22) |
| 세미나실 B | 1건 (H12) | 4건 (H01, H06, H16, H19*) | **5건** | 10-19 (H19) |
| 체력단련실 | 1건 (H07) | 4건 (H03, H11, H17, H20*) | **5건** | 10-21 (H20) |
| 사이버지식정보방 | 1건 (H15) | 1건 (H08) | **2건** | 10-05 (H15) |
| **전체 합계** | **7건** | **15건** | **22건** | - |

*(대대 강당 고장신고 4건, 정기 2건으로 합계 6건 확인 완료)*

#### 2. 미완료 건 목록 (3건)
| 관리번호 | 발생일자 | 시설명 | 고장 증상 | 처리부서 | 현재 상태 | 소요일 |
| :---: | :---: | :--- | :--- | :---: | :---: | :---: |
| H19 | 10-19 | 세미나실 B | 에어컨 냉기 약함 | 군수과 | 처리 중 | - |
| H20 | 10-21 | 체력단련실 | 러닝머신 1호 벨트 이탈 | 군수과 | 처리 중 | - |
| H21 | 10-22 | 대대 강당 | 무선 마이크 수신 불량 | 정보통신실 | 접수 | - |

#### 3. 고장신고 처리 소요일 수작업 검산표
- **완료된 고장신고 건수**: 12건 (전체 15건 중 미완료 3건 제외)
- **소요일 총합**: 2 + 4 + 3 + 2 + 3 + 2 + 5 + 6 + 3 + 4 + 4 + 2 = **39일**
- **평균 처리 소요일**: 39일 ÷ 12건 = **3.25일 (약 3.3일)**
- [x] AI 통계 결과와 수기 교차 검산 일치 확인 완료
"""

print(f"미션 05 실습 기록 완료: 프롬프트 {len(mission_05_prompt.strip())}자, 결과 {len(mission_05_result.strip())}자")


In [ ]:
# [데이터 교차 검산 - 미션5] 정기점검/고장신고 건수 및 완료 건 평균 소요일 계산
summary_m5 = df_maintenance.groupby(['facility', 'type']).size().unstack(fill_value=0)
summary_m5['합계'] = summary_m5.sum(axis=1)
print("=== 시설별 이력 건수 집계 ===")
print(summary_m5)

completed_troubles = df_maintenance[(df_maintenance['type'] == '고장신고') & (df_maintenance['status'] == '완료')]
total_days = completed_troubles['days'].astype(float).sum()
avg_days = completed_troubles['days'].astype(float).mean()

print(f"\n완료된 고장신고 건수: {len(completed_troubles)}건")
print(f"소요일 총합: {total_days:.0f}일, 평균 소요일: {avg_days:.2f}일 (약 {round(avg_days, 1)}일)")
